# Fiverr Market Intelligence — Feature Engineering Pipeline

**Input:** `summary.csv` (flat, one row per gig) + `all_gigs.json` (nested companion: packages, sampled reviews, technologies, seller profiles, media — same 294 gig_ids).
**Output:** Delta table `demo.market_intelligence.fiverr_gig_market_features` (one row per gig) + `demo.market_intelligence.fiverr_technology_metrics`.

### Analytical constraints honored throughout

* **Single snapshot** (scraped 2026-11-15): no growth/decline claims, no time trends beyond sampled review dates.
* **Review records are a sample** (loaded vs total count) — all buyer/price metrics are labeled *sampled* and do not represent all transactions.
* **Gig ratings count = historical demand proxy**, not exact orders sold.
* **Package prices are listing prices**, not transaction prices. No revenue is calculated.
* The dataset has no impressions/clicks/conversions — none are invented.
* No pandas, no `toPandas()` — native Spark only. Raw inputs are never overwritten.

### Sections

1. Configuration and imports · 2. Load CSV · 3. Schema inspection · 4. Data quality checks · 5. Flatten and normalize (+ demand features) · 6. Gig classification · 7. Review feature engineering · 8. Package feature engineering · 9. Seller, competition and technology features (+ opportunity scores) · 10. Final feature table · 11. Validation · 12. Delta table creation · 13. Dashboard SQL queries

## 1. Configuration and imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ---------------------------------------------------------------------------
# 1. CONFIGURATION
# ---------------------------------------------------------------------------

# Input sources (READ-ONLY - this pipeline never writes to these paths)
INPUT_PATH = "/Volumes/demo/bronze/fiverr/summary.csv"        # flat CSV, one row per gig
DETAIL_PATH = "/Volumes/demo/bronze/fiverr/all_gigs.json"     # nested JSON companion (same 294 gig_ids)

# Outputs
OUTPUT_TABLE = "demo.market_intelligence.fiverr_gig_market_features"
TECH_METRICS_TABLE = "demo.market_intelligence.fiverr_technology_metrics"
EXPORT_CSV = True
CSV_EXPORT_PATH = "/Volumes/demo/bronze/fiverr/exports/fiverr_gig_market_features_csv"

BUSINESS_KEY = "gig_id"

print("Input CSV   :", INPUT_PATH)
print("Detail JSON :", DETAIL_PATH)
print("Main output :", OUTPUT_TABLE)

Input CSV   : /Volumes/demo/bronze/fiverr/summary.csv
Detail JSON : /Volumes/demo/bronze/fiverr/all_gigs.json
Main output : demo.market_intelligence.fiverr_gig_market_features


## 2. Load CSV (and nested JSON companion)

The flat `summary.csv` carries one row per gig (listing, seller summary, package prices). The nested companion `all_gigs.json` covers the **same 294 gig_ids** and adds everything the CSV does not contain: full package details (delivery hours, revisions, custom extras), sampled customer reviews, seller profiles, technologies, media and portfolio counts. The CSV is the declared input; the JSON is joined to satisfy the required review, package and technology features. Nothing is fabricated — unknown values stay null.

In [0]:
raw_csv = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .csv(INPUT_PATH))

raw_detail = (spark.read
    .option("multiLine", True)
    .json(DETAIL_PATH))

n_csv, n_json = raw_csv.count(), raw_detail.count()

csv_ids = raw_csv.select(F.col("gig_id").cast("long").alias("gig_id")).distinct()
json_ids = raw_detail.select(F.col("gig.id").cast("long").alias("gig_id")).distinct()

print(f"CSV rows: {n_csv} | JSON rows: {n_json}")
print(f"distinct gig_id  CSV: {csv_ids.count()} | JSON: {json_ids.count()}")
print(f"gig_id overlap CSV<->JSON: {csv_ids.join(json_ids, 'gig_id').count()}")
print(f"CSV-only gig_ids: {csv_ids.join(json_ids, 'gig_id', 'left_anti').count()}")
print(f"JSON-only gig_ids: {json_ids.join(csv_ids, 'gig_id', 'left_anti').count()}")

CSV rows: 294 | JSON rows: 294
distinct gig_id  CSV: 294 | JSON: 294
gig_id overlap CSV<->JSON: 294
CSV-only gig_ids: 0
JSON-only gig_ids: 0


## 3. Schema inspection

Structure is inspected before any transformation — no CSV layout is assumed.

In [0]:
print(f"CSV columns ({len(raw_csv.columns)}):\n{raw_csv.columns}\n")
raw_csv.printSchema()
display(raw_csv.limit(5))

CSV columns (23):
['gig_id', 'url', 'title', 'category', 'sub_category', 'seller_username', 'seller_name', 'seller_level', 'seller_country', 'rating', 'ratings_count', 'seller_completed_orders', 'orders_in_queue', 'basic_price_usd', 'basic_days', 'standard_price_usd', 'standard_days', 'premium_price_usd', 'premium_days', 'tags', 'is_promoted', 'listing_page', 'listing_position']

root
 |-- gig_id: integer (nullable = true)
 |-- url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- seller_username: string (nullable = true)
 |-- seller_name: string (nullable = true)
 |-- seller_level: string (nullable = true)
 |-- seller_country: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- ratings_count: integer (nullable = true)
 |-- seller_completed_orders: integer (nullable = true)
 |-- orders_in_queue: integer (nullable = true)
 |-- basic_price_usd: double (nullable = true)
 |

gig_id,url,title,category,sub_category,seller_username,seller_name,seller_level,seller_country,rating,ratings_count,seller_completed_orders,orders_in_queue,basic_price_usd,basic_days,standard_price_usd,standard_days,premium_price_usd,premium_days,tags,is_promoted,listing_page,listing_position
329808084,https://www.fiverr.com/nicknehm/do-statistical-analysis-for-you,"do statistical analysis for you in rstudio, jasp and python",Data,Data Analytics,nicknehm,Nick,LEVEL_TWO,DE,5.0,40,61,0,60.0,2.0,120.0,3.0,240.0,5.0,rstudio; statistics; hypothesis testing,true,1,0
326305017,https://www.fiverr.com/demigod299/do-expert-data-analysis-and-reporting-spss-stata-minitab,do medical research data analysis and statistical data analysis using spss,Data,Data Analytics,demigod299,Dr. Abdullah K,Top Rated Seller,PK,5.0,198,490,0,25.0,4.0,200.0,6.0,300.0,7.0,spss; data analyst; research; medical; statistical data,false,1,1
139181195,https://www.fiverr.com/rafreis/do-statistical-data-analysis-using-spss-amos-or-excel,"do your data analysis in r, spss or python with a clear report",Data,Data Analytics,rafreis,Rafael R PhD,Top Rated Seller,BR,4.9,275,635,1,45.0,1.0,225.0,1.0,350.0,2.0,spss; rstudio,true,1,2
492261929,https://www.fiverr.com/anika_84/do-econometrics-and-statistical-data-analysis-with-spss-stata-r-python-eviews,"do econometrics and statistical data analysis with spss, stata, r, python eviews",Data,Data Analytics,anika_84,Anika,LEVEL_ONE,PK,5.0,5,5,0,40.0,1.0,150.0,3.0,245.0,7.0,spss; report writing; econometrics; stata,false,1,3
449909727,https://www.fiverr.com/mohsin_54/do-data-analysis-in-spss-jamovi-excel-minitab-and-report-writing,"do data analysis in spss, jamovi, excel, minitab and report writing",Data,Data Analytics,mohsin_54,Data Analyst,Top Rated Seller,PK,5.0,7,352,0,100.0,1.0,120.0,2.0,150.0,2.0,spss; statistics,false,1,4


In [0]:
print(f"JSON top-level columns ({len(raw_detail.columns)}):\n{raw_detail.columns}\n")

for p in [
    "gig.id", "gig.title", "gig.description_text", "gig.rating", "gig.ratings_count",
    "gig.orders_in_queue", "gig.saved_by_count", "gig.tags", "gig.metadata.technology",
    "gig.price_range_usd.low", "gig.price_range_usd.high",
    "seller.username", "seller.level", "seller.country_code", "seller.member_since",
    "listing.position", "listing.page", "listing.is_promoted", "listing.starting_price",
    "reviews.items", "reviews.top_industries", "consultation.pricing",
    "media.images", "portfolio.total_projects", "scraped_at",
]:
    print(f"{p:40s} {raw_detail.select(p).dtypes[0][1]}")

display(raw_detail.select(
    F.col("gig.id").alias("gig_id"),
    F.col("gig.metadata.technology").alias("technology"),
    F.col("gig.metadata.tools").alias("tools"),
    F.col("gig.metadata.programming_language").alias("languages"),
    F.size("packages").alias("n_packages"),
    F.col("reviews.total_count").alias("review_total_count"),
    F.col("reviews.loaded_count").alias("review_loaded_count"),
    F.col("reviews.top_industries"),
).limit(3))

JSON top-level columns (13):
['breadcrumbs', 'consultation', 'gig', 'listing', 'media', 'misc', 'packages', 'portfolio', 'recurring_discounts', 'reviews', 'scraped_at', 'seller', 'url']

gig.id                                   bigint
gig.title                                string
gig.description_text                     string
gig.rating                               double
gig.ratings_count                        bigint
gig.orders_in_queue                      bigint
gig.saved_by_count                       bigint
gig.tags                                 array<string>
gig.metadata.technology                  array<string>
gig.price_range_usd.low                  double
gig.price_range_usd.high                 double
seller.username                          string
seller.level                             string
seller.country_code                      string
seller.member_since                      string
listing.position                         bigint
listing.page                   

gig_id,technology,tools,languages,n_packages,review_total_count,review_loaded_count,top_industries
329808084,"List(Excel, Jupyter Notebook, Other)","List(RStudio, Other)","List(Python, R, SPSS)",3,40,5,"List(1, null, null, null, 2, null, null, null, null, 1, null, null, null, null, null, null, null, null, null, 2, null, null, 1, null, null, null, null, null, null, null, null, null, null)"
326305017,"List(Excel, Google Analytics, Google Sheets, Stata, Other)",List(Other),List(SPSS),3,198,5,"List(null, null, null, null, null, 3, null, null, 1, null, null, null, null, null, null, null, null, null, null, null, null, null, 2, null, null, 1, null, null, 2, null, null, null, null)"
139181195,"List(Google Analytics, Jupyter Notebook, Stata, SPSS, Other)","List(AMOS, SmartPLS, RStudio, Stata, Other)","List(Python, R, SPSS, SQL)",3,275,5,"List(2, 2, null, null, null, 19, null, 1, 1, 5, null, null, 2, null, null, null, 1, 7, 1, 8, 1, null, 6, null, 1, 2, null, 1, 5, null, null, null, null)"


## 4. Data quality and validation

`gig_id` is the main business key. Checks: total records, duplicate gig ids, nulls in important columns, invalid prices, invalid ratings, missing seller ids, missing package/review information, invalid review dates, inconsistent category values, and CSV↔JSON consistency. Nothing is silently repaired — findings are reported below.

In [0]:
flat_detail = raw_detail.select(
    F.col("gig.id").cast("long").alias("gig_id"),
    F.col("seller.id").alias("seller_id"),
    F.col("gig.ratings_count").alias("json_review_count"),
    F.col("reviews.total_count").alias("json_review_total"),
    F.col("reviews.loaded_count").alias("json_review_loaded"),
    F.size("packages").alias("json_package_count"),
    F.to_date(F.substring("scraped_at", 1, 10)).alias("snapshot_date"),
)

json_basic = (raw_detail
    .select(F.col("gig.id").cast("long").alias("gig_id"), F.explode("packages").alias("p"))
    .where(F.lower("p.tier") == "basic")
    .groupBy("gig_id").agg(F.max("p.price_usd").alias("json_basic_price")))

combined = (raw_csv
    .select(F.col("gig_id").cast("long").alias("gig_id"), "title", "category", "sub_category",
            "seller_username", "rating", "ratings_count",
            "basic_price_usd", "standard_price_usd", "premium_price_usd")
    .join(flat_detail, "gig_id", "full_outer")
    .join(json_basic, "gig_id", "left"))

review_dq = (raw_detail
    .select(F.col("gig.id").alias("gig_id"),
            F.to_date(F.substring("scraped_at", 1, 10)).alias("snapshot_date"),
            F.explode("reviews.items").alias("r"))
    .select("gig_id", "snapshot_date",
            F.to_date(F.substring("r.date", 1, 10)).alias("review_date")))

n_invalid_review_dates = review_dq.where(
    F.col("review_date").isNull()
    | (F.col("review_date") < F.lit("2010-01-01").cast("date"))
    | (F.col("review_date") > F.col("snapshot_date"))).count()

n_csv_only = csv_ids.join(json_ids, "gig_id", "left_anti").count()

dq_rows = []

def dq_row(check, metric, affected, status=None):
    dq_rows.append((check, metric, int(affected), status or ("PASS" if affected == 0 else "REVIEW")))

dq_row("total_records", f"{combined.count()} rows loaded (CSV {n_csv} + detail JSON {n_json})",
       combined.count(), "INFO")
dq_row("duplicate_gig_ids", "rows sharing a gig_id with another row",
       n_csv - csv_ids.count())
dq_row("null_gig_id", "gig_id missing",
       combined.where(F.col("gig_id").isNull()).count())
dq_row("null_title", "title missing",
       combined.where(F.col("title").isNull()).count())
dq_row("null_rating", "gig rating missing",
       combined.where(F.col("rating").isNull()).count())
dq_row("invalid_rating", "rating outside 0-5",
       combined.where(F.col("rating").isNotNull() & ~F.col("rating").between(0, 5)).count())
dq_row("null_review_count", "review (ratings) count missing",
       combined.where(F.col("ratings_count").isNull()).count())
dq_row("missing_seller_id", "seller_username or seller.id missing",
       combined.where(F.col("seller_username").isNull() | F.col("seller_id").isNull()).count())
dq_row("missing_package_info", "all basic/standard/premium prices null",
       combined.where(F.col("basic_price_usd").isNull() & F.col("standard_price_usd").isNull()
                     & F.col("premium_price_usd").isNull()).count())
dq_row("invalid_prices", "basic_price_usd <= 0",
       combined.where(F.col("basic_price_usd") <= 0).count())
dq_row("missing_review_info", "ratings_count or reviews.total_count null",
       combined.where(F.col("ratings_count").isNull() | F.col("json_review_total").isNull()).count())
dq_row("invalid_review_dates", "sampled review date null, before 2010, or after snapshot",
       n_invalid_review_dates)
dq_row("inconsistent_category_values", "category <> 'Data' or sub_category <> 'Data Analytics'",
       combined.where((F.col("category") != "Data") | (F.col("sub_category") != "Data Analytics")).count())
dq_row("csv_vs_json_price_mismatch", "CSV basic price differs from JSON basic package price",
       combined.where(F.col("basic_price_usd").isNotNull() & F.col("json_basic_price").isNotNull()
                     & (F.abs(F.col("basic_price_usd") - F.col("json_basic_price")) > 0.01)).count())
dq_row("gig_id_coverage_csv_vs_json", "CSV gig_ids missing from JSON detail", n_csv_only)

print(f"total sampled review records checked: {review_dq.count()}")

dq_report = spark.createDataFrame(dq_rows, "check STRING, metric STRING, records_affected LONG, status STRING")
display(dq_report)

total sampled review records checked: 1241


check,metric,records_affected,status
total_records,294 rows loaded (CSV 294 + detail JSON 294),294,INFO
duplicate_gig_ids,rows sharing a gig_id with another row,0,PASS
null_gig_id,gig_id missing,0,PASS
null_title,title missing,0,PASS
null_rating,gig rating missing,0,PASS
invalid_rating,rating outside 0-5,0,PASS
null_review_count,review (ratings) count missing,0,PASS
missing_seller_id,seller_username or seller.id missing,0,PASS
missing_package_info,all basic/standard/premium prices null,0,PASS
invalid_prices,basic_price_usd <= 0,0,PASS


## 5. Flatten and normalize

One row per gig built from the nested detail, deduplicated on `gig_id`. Token lists (tags, technologies, programming languages, analysis types, expertise, tools) are normalized: lowercase, alias canonicalization, deduplicated, sorted. Demand features are computed here too.

Notes:
* `category` / `sub_category` are constant across this snapshot ("Data" / "Data Analytics"), so **competition metrics later use `nested_sub_category` as the meaningful category scope**.
* Review counts are a **historical demand proxy** — not exact orders sold.

In [0]:
def norm_text(c):
    """lowercase, keep alphanumeric/+#. characters, collapse whitespace"""
    c = F.lower(F.coalesce(c, F.lit("")))
    c = F.regexp_replace(c, "[^a-z0-9+#. ]", " ")
    return F.trim(F.regexp_replace(c, " +", " "))

TECH_ALIASES = {
    "powerbi": "power bi", "power bi pro": "power bi", "microsoft power bi": "power bi",
    "microsoft excel": "excel", "ms excel": "excel",
    "r studio": "rstudio", "r programming": "r",
    "apache spark": "spark", "pyspark": "spark",
    "google data studio": "looker studio",
    "ml": "machine learning",
}

def apply_alias(c):
    out = c
    for alias, canon in TECH_ALIASES.items():
        out = F.when(c == F.lit(alias), F.lit(canon)).otherwise(out)
    return out

def normalize_list(array_col):
    arr = F.coalesce(array_col, F.array())
    normed = F.transform(arr, lambda x: apply_alias(norm_text(x)))
    normed = F.filter(normed, lambda x: F.length(x) > 0)
    return F.array_sort(F.array_distinct(normed))

gig_base = raw_detail.select(
    # --- identity
    F.col("gig.id").cast("long").alias("gig_id"),
    F.col("url").alias("gig_url"),
    F.col("gig.title").alias("title"),
    F.col("gig.description_text").alias("description_text"),
    F.col("gig.category.name").alias("category"),
    F.col("gig.sub_category.name").alias("sub_category"),
    F.col("gig.nested_sub_category.name").alias("nested_sub_category"),
    # --- listing and flags
    F.col("listing.position").cast("long").alias("listing_position"),
    F.col("listing.page").cast("long").alias("listing_page"),
    F.coalesce(F.col("listing.is_promoted"), F.lit(False)).alias("is_promoted"),
    F.coalesce(F.col("gig.is_pro"), F.col("listing.is_pro"), F.lit(False)).alias("is_pro"),
    F.coalesce(F.col("gig.is_handpicked"), F.lit(False)).alias("is_handpicked"),
    F.col("listing.starting_price").cast("double").alias("starting_price"),
    F.col("gig.price_range_usd.low").cast("double").alias("price_range_low"),
    F.col("gig.price_range_usd.high").cast("double").alias("price_range_high"),
    # --- demand
    F.col("gig.rating").cast("double").alias("gig_rating"),
    F.col("gig.ratings_count").cast("long").alias("gig_review_count"),
    F.col("gig.orders_in_queue").cast("long").alias("orders_in_queue"),
    F.col("gig.saved_by_count").cast("long").alias("saved_by_count"),
    # --- seller
    F.col("seller.id").alias("seller_id"),
    F.col("seller.username").alias("seller_username"),
    F.col("seller.country_code").alias("seller_country"),
    F.col("seller.level").alias("seller_level"),
    F.to_date(F.substring("seller.member_since", 1, 10)).alias("seller_member_since"),
    F.col("seller.response_time_hours").cast("double").alias("seller_response_time_hours"),
    F.col("seller.hourly_rate_usd").cast("double").alias("seller_hourly_rate"),
    F.col("seller.rating").cast("double").alias("seller_rating"),
    F.col("seller.ratings_count").cast("long").alias("seller_ratings_count"),
    F.col("seller.completed_orders").cast("long").alias("seller_completed_orders"),
    # --- content and skills
    normalize_list(F.col("gig.tags")).alias("tag_list_arr"),
    normalize_list(F.col("gig.metadata.technology")).alias("technology_list_arr"),
    normalize_list(F.col("gig.metadata.programming_language")).alias("programming_language_list_arr"),
    normalize_list(F.col("gig.metadata.analysis_type")).alias("analysis_type_list_arr"),
    normalize_list(F.col("gig.metadata.expertise")).alias("expertise_list_arr"),
    normalize_list(F.col("gig.metadata.tools")).alias("tools_list_arr"),
    F.when(F.length(F.coalesce(F.col("gig.description_text"), F.lit(""))) > 0,
           F.size(F.split(F.trim(F.col("gig.description_text")), " +"))).cast("long").alias("description_word_count"),
    F.size(F.split(F.trim(F.coalesce(F.col("gig.title"), F.lit(""))), " +")).cast("long").alias("title_word_count"),
    F.coalesce(F.size(F.col("media.images")), F.lit(0)).cast("long").alias("image_count"),
    F.coalesce(F.size(F.col("media.videos")), F.lit(0)).cast("long").alias("video_count"),
    F.col("portfolio.total_projects").cast("long").alias("portfolio_project_count"),
    # --- nested arrays for later sections
    F.col("packages").alias("gig_packages"),
    F.col("reviews.items").alias("review_items"),
    F.col("reviews.total_count").cast("long").alias("review_total_count"),
    F.col("reviews.loaded_count").cast("long").alias("review_loaded_count"),
    F.col("reviews.top_industries").alias("review_top_industries"),
    F.col("consultation.pricing").alias("consultation_pricing"),
    F.to_date(F.substring("scraped_at", 1, 10)).alias("snapshot_date"),
)

# Guard: exactly one row per gig_id (keep first listing occurrence)
w_dedupe = Window.partitionBy("gig_id").orderBy("listing_page", "listing_position")
gig_base = (gig_base
    .withColumn("_rn", F.row_number().over(w_dedupe))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

print(f"gig_base rows: {gig_base.count()} | distinct gig_id: {gig_base.select('gig_id').distinct().count()}")

gig_base rows: 294 | distinct gig_id: 294


In [0]:
# Demand features - review counts are a HISTORICAL DEMAND PROXY (not orders sold)
demand_features = gig_base.select(
    "gig_id",
    F.round(F.log1p("gig_review_count"), 4).alias("log_review_count"),
    F.round(F.percent_rank().over(
        Window.orderBy(F.coalesce(F.col("gig_review_count"), F.lit(-1)))), 4).alias("demand_percentile"),
    (F.when(F.col("gig_review_count") == 0, F.lit("0 reviews"))
      .when(F.col("gig_review_count").between(1, 10), F.lit("1-10"))
      .when(F.col("gig_review_count").between(11, 50), F.lit("11-50"))
      .when(F.col("gig_review_count").between(51, 100), F.lit("51-100"))
      .when(F.col("gig_review_count").between(101, 500), F.lit("101-500"))
      .when(F.col("gig_review_count") > 500, F.lit("500+"))
      .otherwise(F.lit("unknown"))).alias("gig_review_band"),
)

display(demand_features.groupBy("gig_review_band")
    .agg(F.count("*").alias("gigs"))
    .orderBy("gig_review_band"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gig_review_band,gigs
0 reviews,9
1-10,137
101-500,13
11-50,110
500+,4
51-100,21


## 6. Gig classification

Multi-label keyword rules over **five evidence sources** (title, tags, description, metadata, nested category) — never the nested category alone. The primary `service_family` is the family with the **strongest evidence** (most matching sources); ties are broken by the priority order below.

* `service_labels` — all matched families (multi-label, comma-separated)
* `classification_confidence` — share of the 5 evidence sources that matched the primary family
* `needs_manual_review` — no family matched OR confidence < 0.4

In [0]:
FAMILY_RULES = {
    "BI & Dashboarding": [
        "dashboard", "power bi", "powerbi", "tableau", "looker", "business intelligence",
        "data studio", "kpi", "visualization", "visualize", "visualisation",
    ],
    "Statistical & Research Analysis": [
        "statistical", "statistics", "spss", "stata", "anova", "regression", "hypothesis",
        "econometric", "survey", "research", "thesis", "dissertation", "qualitative",
        "quantitative", "nvivo", "maxqda", "minitab", "jamovi", "rstudio", "smart pls",
        "t test", "correlation", "descriptive", "inferential",
    ],
    "Financial & Business Analytics": [
        "financial", "finance", "accounting", "bookkeeping", "book keeping", "balance sheet",
        "profit", "valuation", "investment", "stock", "trading", "cash flow", "budget",
        "financial statement", "quickbooks",
    ],
    "Data Science & Machine Learning": [
        "machine learning", "deep learning", "neural", "nlp", "predictive", "artificial intelligence",
        "tensorflow", "pytorch", "scikit", "sklearn", "clustering", "recommendation", "chatbot",
        "computer vision", "data science", "sentiment analysis",
    ],
    "Data Engineering & Data Pipelines": [
        "pipeline", "etl", "elt", "data engineer", "airflow", "kafka", "dbt", "snowflake",
        "databricks", "warehouse", "ingestion", "spark", "hadoop", "big data", "data lake",
    ],
    "Marketing & Web Analytics": [
        "marketing", "seo", "google analytics", "ga4", "ads", "social media", "web analytics",
        "conversion", "funnel", "tiktok", "instagram", "facebook", "campaign", "advertis",
    ],
    "Product & Customer Analytics": [
        "product analytics", "mixpanel", "amplitude", "churn", "retention", "cohort",
        "customer analytic", "user behavior", "segmentation", "customer insight",
    ],
    "Supply Chain & Operations Analytics": [
        "supply chain", "logistics", "inventory", "operations", "demand planning",
        "procurement", "warehouse management", "scheduling optimization",
    ],
    "Data Cleaning & Processing": [
        "data cleaning", "data entry", "clean up", "cleanup", "data processing",
        "data collection", "formatting", "data preparation", "mining", "scraping",
    ],
    "Data Consultation & Tutoring": [
        "consult", "tutor", "teaching", "training", "coaching", "lesson", "phd",
        "guidance", "mentor", "academic support",
    ],
}
FAMILY_PRIORITY = list(FAMILY_RULES.keys())
EVIDENCE_SOURCES = ["title_lc", "tags_lc", "desc_lc", "meta_lc", "nested_lc"]

def has_any(colname, keywords):
    checks = [F.instr(F.col(colname), F.lit(kw)) > 0 for kw in keywords]
    return checks[0] if len(checks) == 1 else F.greatest(*checks)

cls = (gig_base.select(
        "gig_id", "title", "description_text", "nested_sub_category",
        "tag_list_arr", "technology_list_arr", "programming_language_list_arr",
        "tools_list_arr", "analysis_type_list_arr", "expertise_list_arr",
    )
    .withColumn("title_lc", F.lower(F.coalesce("title", F.lit(""))))
    .withColumn("tags_lc", F.concat_ws(" ", "tag_list_arr"))
    .withColumn("desc_lc", F.lower(F.coalesce("description_text", F.lit(""))))
    .withColumn("meta_lc", F.concat_ws(" ", "technology_list_arr", "programming_language_list_arr",
                                        "tools_list_arr", "analysis_type_list_arr", "expertise_list_arr"))
    .withColumn("nested_lc", F.lower(F.coalesce("nested_sub_category", F.lit(""))))
    .withColumn("full_lc", F.concat_ws(" ", "title_lc", "tags_lc", "desc_lc", "meta_lc", "nested_lc")))

for i, fam in enumerate(FAMILY_PRIORITY):
    kws = FAMILY_RULES[fam]
    cls = cls.withColumn(f"_match_{i}", F.when(has_any("full_lc", kws), F.lit(1)).otherwise(F.lit(0)))
    evidence = None
    for src in EVIDENCE_SOURCES:
        c = F.when(has_any(src, kws), F.lit(1)).otherwise(F.lit(0))
        evidence = c if evidence is None else evidence + c
    cls = cls.withColumn(f"_evidence_{i}", F.round(evidence / F.lit(len(EVIDENCE_SOURCES)), 3))

label_array = F.filter(
    F.array(*[F.when(F.col(f"_match_{i}") == 1, F.lit(fam)) for i, fam in enumerate(FAMILY_PRIORITY)]),
    lambda x: x.isNotNull(),
)

# Primary family = the family with the strongest evidence (most matching
# sources); ties are broken by the family priority order above.
max_evidence = F.greatest(*[F.col(f"_evidence_{i}") for i in range(len(FAMILY_PRIORITY))])

cls = (cls
    .withColumn("service_labels", label_array)
    .withColumn("service_family", F.coalesce(
        *[F.when((F.col(f"_match_{i}") == 1) & (F.col(f"_evidence_{i}") == max_evidence), F.lit(fam))
          for i, fam in enumerate(FAMILY_PRIORITY)],
        F.lit("Other / Needs Review")))
    .withColumn("classification_confidence", F.coalesce(*[
        F.when(F.col("service_family") == F.lit(fam), F.col(f"_evidence_{i}"))
        for i, fam in enumerate(FAMILY_PRIORITY)
    ]))
    .withColumn("classification_confidence", F.coalesce(F.col("classification_confidence"), F.lit(0.0)))
    .withColumn("needs_manual_review",
        (F.col("service_family") == F.lit("Other / Needs Review"))
        | (F.col("classification_confidence") < 0.4)))

gig_class = cls.select("gig_id", "service_family", "service_labels",
                       "classification_confidence", "needs_manual_review")

print(f"classified gigs: {gig_class.count()} | distinct families used: {gig_class.select('service_family').distinct().count()}")

classified gigs: 294 | distinct families used: 10


In [0]:
display(gig_class.groupBy("service_family")
    .agg(F.count("*").alias("gigs"),
         F.round(F.avg("classification_confidence"), 3).alias("avg_confidence"),
         F.sum(F.col("needs_manual_review").cast("long")).alias("needs_review"))
    .orderBy(F.desc("gigs")))

service_family,gigs,avg_confidence,needs_review
Statistical & Research Analysis,171,0.841,5
BI & Dashboarding,69,0.67,2
Data Science & Machine Learning,12,0.7,0
Financial & Business Analytics,11,0.818,0
Marketing & Web Analytics,10,0.74,0
Product & Customer Analytics,8,0.8,0
Data Consultation & Tutoring,6,0.633,0
Supply Chain & Operations Analytics,4,0.45,0
Data Engineering & Data Pipelines,2,0.8,0
Data Cleaning & Processing,1,0.6,0


## 7. Review feature engineering (SAMPLED reviews only)

The dataset contains **sampled** review records per gig — these metrics describe the available sample, not all transactions or all historical customers. Review price ranges are parsed to estimated midpoints (e.g. "Up to $50" → $25, "$50–$100" → $75, "$200–$400" → $300); **unknown values stay null**. No revenue is computed anywhere. Review items are aggregated per gig_id before joining to the main table.

In [0]:
review_exploded = (gig_base
    .where(F.col("review_items").isNotNull())
    .select("gig_id", F.explode("review_items").alias("r")))

# --- Parse order price range to an estimated midpoint ----------------------
# "$200-$400" -> 300, "Up to $50" -> 25, "$800+" -> 800, unknown -> null
pr_txt = F.regexp_replace(F.lower(F.coalesce(F.col("r.order_price_range_usd"), F.lit(""))), ",", "")
lower_val = F.regexp_extract(pr_txt, "[$]([0-9]+)", 1)
upper_val = F.regexp_extract(pr_txt, "[-][$]([0-9]+)", 1)

price_midpoint = (
    F.when((F.length(lower_val) > 0) & (F.length(upper_val) > 0),
           (lower_val.cast("double") + upper_val.cast("double")) / 2)
     .when((F.length(lower_val) > 0) & (F.length(upper_val) == 0) & pr_txt.contains("up to"),
           lower_val.cast("double") / 2)
     .when((F.length(lower_val) > 0) & (F.length(upper_val) == 0) & pr_txt.contains("+"),
           lower_val.cast("double"))
     .otherwise(F.lit(None).cast("double"))
)

review_agg = (review_exploded
    .withColumn("price_midpoint", price_midpoint)
    .withColumn("review_date", F.to_date(F.substring(F.col("r.date"), 1, 10)))
    .groupBy("gig_id")
    .agg(
        F.count("*").cast("long").alias("sampled_review_count"),
        F.round(F.avg("r.rating"), 3).alias("avg_review_rating"),
        F.percentile_approx("r.rating", 0.5).cast("double").alias("median_review_rating"),
        F.round(F.avg(F.when(F.col("r.rating") >= 4, F.lit(1.0)).otherwise(F.lit(0.0))), 4).alias("positive_review_share"),
        F.round(F.avg(F.col("r.is_business").cast("double")), 4).alias("business_buyer_share"),
        F.round(F.avg(F.col("r.repeat_buyer").cast("double")), 4).alias("repeat_buyer_share"),
        F.percentile_approx(F.col("r.order_duration_days").cast("double"), 0.5).cast("double").alias("median_order_duration_days"),
        F.percentile_approx("price_midpoint", 0.5).cast("double").alias("review_order_price_midpoint_median"),
        F.round(F.avg(F.when(F.col("price_midpoint").isNotNull(),
                             F.when(F.col("price_midpoint") >= 200, F.lit(1.0)).otherwise(F.lit(0.0)))), 4).alias("buyer_price_200_plus_share"),
        F.max("review_date").alias("latest_sampled_review_date"),
    ))

# --- Most frequent sampled buyer country per gig ---------------------------
top_countries = (review_exploded
    .where(F.col("r.reviewer_country").isNotNull())
    .groupBy("gig_id", F.col("r.reviewer_country").alias("buyer_country"))
    .agg(F.count("*").alias("n_reviews"))
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("gig_id").orderBy(F.desc("n_reviews"), F.desc("buyer_country"))))
    .filter(F.col("rn") == 1)
    .select("gig_id", F.col("buyer_country").alias("top_buyer_country")))

# --- Top buyer industry from the gig-level industries struct ---------------
industry_names = raw_detail.select("reviews.top_industries.*").columns
entries = F.array(*[
    F.struct(F.lit(n).alias("k"),
             F.coalesce(F.col(f"review_top_industries.{n}"), F.lit(0).cast("long")).alias("v"))
    for n in industry_names
])
top_entry = F.element_at(
    F.reverse(F.array_sort(F.transform(entries, lambda e: F.struct(e.v.alias("v"), e.k.alias("k"))))), 1)
top_industry = F.when(top_entry.v > 0, top_entry.k)

review_features = (gig_base
    .select(
        "gig_id", "snapshot_date",
        F.round(F.when(F.col("review_total_count") > 0,
                       F.col("review_loaded_count") / F.col("review_total_count")), 4).alias("review_coverage_ratio"),
        top_industry.alias("top_buyer_industry"),
    )
    .join(review_agg, "gig_id", "left")
    .join(top_countries, "gig_id", "left")
    .withColumn("sampled_review_recency_days",
        F.when(F.col("latest_sampled_review_date").isNotNull(),
               F.datediff("snapshot_date", "latest_sampled_review_date")))
    .drop("snapshot_date"))

print(f"gigs with sampled reviews: {review_features.where(F.col('sampled_review_count') > 0).count()}")
display(review_features.select("gig_id", "sampled_review_count", "business_buyer_share", "repeat_buyer_share",
                                "review_order_price_midpoint_median", "top_buyer_country", "top_buyer_industry",
                                "latest_sampled_review_date", "sampled_review_recency_days", "review_coverage_ratio").limit(5))

gigs with sampled reviews: 285


gig_id,sampled_review_count,business_buyer_share,repeat_buyer_share,review_order_price_midpoint_median,top_buyer_country,top_buyer_industry,latest_sampled_review_date,sampled_review_recency_days,review_coverage_ratio
11472347,5,0.4,0.0,25.0,United States,financial_services_or_banking,2026-07-16,122,0.0087
110678455,4,0.0,0.0,75.0,United States,technology,2025-10-23,388,1.0
116697469,5,0.0,0.6,75.0,United States,technology,2026-09-24,52,0.0182
122062614,5,0.2,0.6,null,Brazil,financial_services_or_banking,2023-04-10,1315,0.8333
125318834,5,0.0,0.6,300.0,United States,education,2026-02-16,272,0.1923


## 8. Package feature engineering

Package arrays are pivoted by tier into basic / standard / premium columns. These are **listing prices, not transaction prices**. `consultation_price` = lowest listed consultation session price. `custom_extra_count` = custom extras summed across packages. Price differences and ratios are computed from these listing prices only.

In [0]:
pkg_exploded = gig_base.select(
    "gig_id", "price_range_low", "price_range_high", "consultation_pricing",
    F.explode("gig_packages").alias("p"),
)

def tier_max(tier, field, dtype):
    return F.max(F.when(F.lower(F.col("p.tier")) == F.lit(tier), F.col(f"p.{field}"))).cast(dtype)

package_agg = pkg_exploded.groupBy("gig_id").agg(
    F.min("p.price_usd").cast("double").alias("min_package_price"),
    F.max("p.price_usd").cast("double").alias("max_package_price"),
    F.percentile_approx("p.price_usd", 0.5).cast("double").alias("median_package_price"),
    tier_max("basic", "price_usd", "double").alias("basic_price"),
    tier_max("standard", "price_usd", "double").alias("standard_price"),
    tier_max("premium", "price_usd", "double").alias("premium_price"),
    tier_max("basic", "delivery_hours", "long").alias("basic_delivery_hours"),
    tier_max("standard", "delivery_hours", "long").alias("standard_delivery_hours"),
    tier_max("premium", "delivery_hours", "long").alias("premium_delivery_hours"),
    tier_max("basic", "revisions", "long").alias("basic_revisions"),
    tier_max("standard", "revisions", "long").alias("standard_revisions"),
    tier_max("premium", "revisions", "long").alias("premium_revisions"),
    F.sum(F.coalesce(F.size("p.custom_extras"), F.lit(0))).cast("long").alias("custom_extra_count"),
)

consultation_price = F.when(
    F.size(F.coalesce(F.col("consultation_pricing"), F.array())) > 0,
    F.array_min(F.transform(F.col("consultation_pricing"), lambda p: p.price.cast("double"))),
)

package_features = (package_agg
    .join(gig_base.select("gig_id", "price_range_low", "price_range_high",
                          consultation_price.alias("consultation_price")),
          "gig_id", "right")
    .select(
        "gig_id",
        F.coalesce("price_range_low", "min_package_price").alias("gig_min_price"),
        F.coalesce("price_range_high", "max_package_price").alias("gig_max_price"),
        "basic_price", "standard_price", "premium_price", "median_package_price",
        "basic_delivery_hours", "standard_delivery_hours", "premium_delivery_hours",
        "basic_revisions", "standard_revisions", "premium_revisions",
        "custom_extra_count", "consultation_price",
    )
    .withColumn("gig_price_spread",
        F.when(F.col("gig_min_price").isNotNull() & F.col("gig_max_price").isNotNull(),
               F.round(F.col("gig_max_price") - F.col("gig_min_price"), 2)))
    .withColumn("package_price_ratio",
        F.when(F.col("basic_price") > 0,
               F.round(F.col("premium_price") / F.col("basic_price"), 3))))

print(f"gigs with package features: {package_features.count()}")
display(package_features.limit(5))

gigs with package features: 294


gig_id,gig_min_price,gig_max_price,basic_price,standard_price,premium_price,median_package_price,basic_delivery_hours,standard_delivery_hours,premium_delivery_hours,basic_revisions,standard_revisions,premium_revisions,custom_extra_count,consultation_price,gig_price_spread,package_price_ratio
11472347,25.0,200.0,25.0,75.0,200.0,75.0,72,144,240,null,null,null,15,null,175.0,8.0
110678455,200.0,600.0,200.0,400.0,600.0,400.0,120,120,120,null,null,null,0,null,400.0,3.0
116697469,25.0,500.0,25.0,125.0,500.0,125.0,48,96,144,null,null,null,15,null,475.0,20.0
122062614,250.0,1000.0,250.0,500.0,1000.0,500.0,72,120,168,2,3,4,6,50.0,750.0,4.0
125318834,380.0,1400.0,380.0,610.0,1400.0,610.0,96,120,336,1,2,4,0,null,1020.0,3.684


## 9. Seller, competition and technology features

* **Technology** — alias-normalized lists plus boolean flags for important technologies and services (keyword-based indicators).
* **Seller** — gig-inherited seller attributes. `seller_gig_count` counts gigs per distinct seller username. When aggregating to market level, use `COUNT(DISTINCT seller_username)` or `MAX` on seller-level values — never `SUM` (a seller's metrics are not summed across their gigs).
* **Competition** — category scope = `nested_sub_category` (category / sub_category are constant in this snapshot). Seller counts use DISTINCT sellers. `category_top10_review_share` = share of category review demand held by the top 10 gigs. `gig_price_vs_category_median` = starting price ÷ category median starting price.

In [0]:
def any_expr(exprs):
    exprs = list(exprs)
    return exprs[0] if len(exprs) == 1 else F.greatest(*exprs)

def has_token(words):
    if isinstance(words, str):
        words = [words]
    return any_expr([F.array_contains(F.col("tech_tokens"), w) for w in words])

def has_phrase(phrases):
    if isinstance(phrases, str):
        phrases = [phrases]
    padded = F.concat(F.lit(" "), F.col("tech_text"), F.lit(" "))
    return any_expr([F.instr(padded, p) > 0 for p in phrases])

tech_source = (gig_base
    .withColumn("tech_text",
        F.concat_ws(" ", "technology_list_arr", "programming_language_list_arr",
                    "tools_list_arr", "tag_list_arr",
                    F.lower(F.coalesce(F.col("title"), F.lit("")))))
    .withColumn("tech_tokens", F.split(F.col("tech_text"), " +")))

tech_features = tech_source.select(
    "gig_id",
    F.size("technology_list_arr").cast("long").alias("technology_count"),
    F.size("tag_list_arr").cast("long").alias("tag_count"),
    has_token("python").alias("python_flag"),
    has_token(["sql", "mysql", "postgresql", "sqlite"]).alias("sql_flag"),
    has_token("excel").alias("excel_flag"),
    has_phrase("power bi").alias("powerbi_flag"),
    has_token("tableau").alias("tableau_flag"),
    has_token("spark").alias("spark_flag"),
    has_token("kafka").alias("kafka_flag"),
    has_token("dbt").alias("dbt_flag"),
    has_token("snowflake").alias("snowflake_flag"),
    has_phrase("machine learning").alias("machine_learning_flag"),
    (has_token(["automation", "automate", "automated", "automating"])
     | has_phrase("automation")).alias("automation_flag"),
    has_phrase("dashboard").alias("dashboard_flag"),
    (has_phrase(["pipeline", "etl", "elt"])
     | has_token(["airflow", "airbyte", "fivetran"])).alias("data_pipeline_flag"),
    has_phrase(["financial", "finance", "accounting", "bookkeeping", "book keeping"]).alias("financial_analytics_flag"),
    has_phrase(["marketing", "seo", "google analytics", "ga4", " ads", "advertis", "campaign"]).alias("marketing_analytics_flag"),
    has_phrase(["product analytic", "mixpanel", "amplitude", "cohort", "churn", "retention", "customer analytic"]).alias("product_analytics_flag"),
)

FLAG_COLUMNS = [c for c in tech_features.columns if c.endswith("_flag")]
display(tech_features.groupBy().agg(*[F.sum(F.col(c).cast("long")).alias(c) for c in FLAG_COLUMNS]))

python_flag,sql_flag,excel_flag,powerbi_flag,tableau_flag,spark_flag,kafka_flag,dbt_flag,snowflake_flag,machine_learning_flag,automation_flag,dashboard_flag,data_pipeline_flag,financial_analytics_flag,marketing_analytics_flag,product_analytics_flag
224,166,254,164,125,21,0,0,1,21,10,40,3,13,137,9


In [0]:
w_seller = Window.partitionBy("seller_username")

seller_features = (gig_base
    .withColumn("seller_gig_count", F.count(F.lit(1)).over(w_seller))
    .withColumn("seller_tenure_months",
        F.when(F.col("seller_member_since").isNotNull() & F.col("snapshot_date").isNotNull(),
               F.round(F.months_between(F.col("snapshot_date"), F.col("seller_member_since")), 1)))
    .select("gig_id", "seller_gig_count", "seller_tenure_months"))

display(seller_features.limit(5))

gig_id,seller_gig_count,seller_tenure_months
416290309,1,72.2
280273454,1,103.3
378193967,1,43.0
252541932,1,74.4
281642842,1,48.9


In [0]:
comp_input = (gig_base
    .select("gig_id", "nested_sub_category", "seller_username", "gig_review_count", "starting_price", "is_promoted")
    .join(gig_class.select("gig_id", "service_family"), "gig_id"))

category_agg = (comp_input.groupBy("nested_sub_category").agg(
    F.count("gig_id").cast("long").alias("category_gig_count"),
    F.countDistinct("seller_username").cast("long").alias("category_seller_count"),
    F.percentile_approx("gig_review_count", 0.5).cast("double").alias("category_review_median"),
    F.percentile_approx("gig_review_count", 0.75).cast("double").alias("category_review_p75"),
    F.percentile_approx("gig_review_count", 0.9).cast("double").alias("category_review_p90"),
    F.percentile_approx("starting_price", 0.5).cast("double").alias("category_price_median"),
    F.round(F.avg(F.col("is_promoted").cast("double")), 4).alias("category_promoted_share"),
))

w_rank = Window.partitionBy("nested_sub_category").orderBy(F.desc("gig_review_count"))
top10_share = (comp_input
    .withColumn("rk", F.row_number().over(w_rank))
    .groupBy("nested_sub_category")
    .agg(F.sum(F.when(F.col("rk") <= 10, F.col("gig_review_count"))).alias("top10_reviews"),
         F.sum("gig_review_count").alias("total_reviews"))
    .select("nested_sub_category",
            F.round(F.col("top10_reviews") / F.col("total_reviews"), 4).alias("category_top10_review_share")))

family_agg = (comp_input.groupBy("service_family").agg(
    F.count("gig_id").cast("long").alias("service_family_gig_count"),
    F.countDistinct("seller_username").cast("long").alias("service_family_seller_count"),
    F.percentile_approx("gig_review_count", 0.5).cast("double").alias("service_family_review_median"),
    F.percentile_approx("starting_price", 0.5).cast("double").alias("service_family_price_median"),
    F.round(F.avg(F.col("is_promoted").cast("double")), 4).alias("service_family_promoted_share"),
))

w_cat = Window.partitionBy("nested_sub_category").orderBy("gig_review_count")

competition_features = (comp_input
    .withColumn("gig_review_percentile", F.round(F.percent_rank().over(w_cat), 4))
    .join(category_agg, "nested_sub_category")
    .join(top10_share, "nested_sub_category")
    .join(family_agg, "service_family")
    .withColumn("gig_price_vs_category_median",
        F.when(F.col("category_price_median") > 0,
               F.round(F.col("starting_price") / F.col("category_price_median"), 3)))
    .select("gig_id", "category_gig_count", "category_seller_count",
            "category_review_median", "category_review_p75", "category_review_p90",
            "category_price_median", "category_promoted_share", "category_top10_review_share",
            "gig_review_percentile", "gig_price_vs_category_median",
            "service_family_gig_count", "service_family_seller_count",
            "service_family_review_median", "service_family_price_median", "service_family_promoted_share"))

print(f"gigs with competition features: {competition_features.count()}")
display(competition_features.limit(5))

gigs with competition features: 294


gig_id,category_gig_count,category_seller_count,category_review_median,category_review_p75,category_review_p90,category_price_median,category_promoted_share,category_top10_review_share,gig_review_percentile,gig_price_vs_category_median,service_family_gig_count,service_family_seller_count,service_family_review_median,service_family_price_median,service_family_promoted_share
281642842,140,121,15.0,36.0,83.0,35.0,0.3071,0.6132,1.0,0.143,171,143,15.0,40.0,0.2924
187853872,33,33,7.0,17.0,23.0,100.0,0.0303,0.7495,1.0,0.65,171,143,15.0,40.0,0.2924
262571196,9,8,9.0,10.0,81.0,100.0,0.3333,1.0,1.0,0.05,8,8,9.0,50.0,0.25
11472347,65,61,9.0,25.0,70.0,50.0,0.1385,0.7392,1.0,0.5,69,67,9.0,65.0,0.0435
405700716,10,9,3.0,12.0,15.0,20.0,0.0,1.0,1.0,3.5,4,3,12.0,30.0,0.0


## 9b. Derived market opportunity scores

Transparent percentile-rank component scores (0–1) over the whole gig population:

* `demand_score` — percentile rank of log review count (demand proxy)
* `competition_score` — percentile rank of category gig count (higher = more crowded)
* `pricing_power_score` — percentile rank of price vs category median
* `business_buyer_score` — percentile rank of sampled business-buyer share (null when no sampled reviews)

`market_opportunity_score` = **0.40·demand + 0.25·(1 − competition) + 0.20·pricing_power + 0.15·business_buyer**

Weights are a stated heuristic, adjustable. This score is **not** an objective probability of market success. All components remain separately available so the score can be understood and re-weighted.

In [0]:
scores = (gig_base.select("gig_id")
    .join(demand_features.select("gig_id", "log_review_count"), "gig_id")
    .join(competition_features.select("gig_id", "category_gig_count", "gig_price_vs_category_median"), "gig_id")
    .join(review_features.select("gig_id", "business_buyer_share"), "gig_id", "left")
    .select(
        "gig_id",
        F.round(F.percent_rank().over(Window.orderBy(F.col("log_review_count"))), 3).alias("demand_score"),
        F.round(F.percent_rank().over(Window.orderBy(F.col("category_gig_count"))), 3).alias("competition_score"),
        F.when(F.col("gig_price_vs_category_median").isNotNull(),
               F.round(F.percent_rank().over(Window.orderBy(F.col("gig_price_vs_category_median"))), 3)).alias("pricing_power_score"),
        F.when(F.col("business_buyer_share").isNotNull(),
               F.round(F.percent_rank().over(Window.orderBy(F.col("business_buyer_share"))), 3)).alias("business_buyer_score"),
    )
    .withColumn("market_opportunity_score", F.round(
        0.40 * F.col("demand_score")
        + 0.25 * (F.lit(1.0) - F.col("competition_score"))
        + 0.20 * F.col("pricing_power_score")
        + 0.15 * F.col("business_buyer_score"), 3)))

display(scores.orderBy(F.desc("market_opportunity_score")).limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gig_id,demand_score,competition_score,pricing_power_score,business_buyer_score,market_opportunity_score
257474689,0.84,0.096,0.969,0.952,0.899
372596899,0.962,0.0,0.294,0.846,0.821
395678418,0.809,0.304,0.997,0.666,0.797
213784834,0.843,0.096,0.433,0.908,0.786
159030612,0.611,0.191,0.99,0.908,0.781
150598687,0.884,0.526,0.898,0.846,0.779
262571196,0.925,0.0,0.0,0.973,0.766
274310948,0.908,0.526,0.908,0.666,0.763
387651276,0.468,0.0,0.918,0.908,0.757
187853872,0.973,0.191,0.328,0.666,0.757


## 10. Final feature table

All feature groups are joined on `gig_id` into one row per gig. Arrays are converted to comma-separated text lists for dashboards. The technology-level aggregate table is built from the array version before flattening. A sample and column documentation follow below the assembly.

In [0]:
final_build = (gig_base
    .join(demand_features, "gig_id")
    .join(gig_class, "gig_id")
    .join(review_features, "gig_id", "left")
    .join(package_features, "gig_id", "left")
    .join(tech_features, "gig_id", "left")
    .join(seller_features, "gig_id", "left")
    .join(competition_features, "gig_id", "left")
    .join(scores, "gig_id", "left"))

print(f"final_build: {final_build.count()} rows, {len(final_build.columns)} columns")

# --- Technology-level aggregates (from array version, before text flattening)
tech_metrics = (final_build
    .select(F.explode("technology_list_arr").alias("technology"),
            "gig_id", "gig_review_count", "starting_price", "is_promoted")
    .groupBy("technology")
    .agg(
        F.count("gig_id").cast("long").alias("gig_count"),
        F.percentile_approx("gig_review_count", 0.5).cast("double").alias("median_gig_review_count"),
        F.percentile_approx("starting_price", 0.5).cast("double").alias("median_starting_price"),
        F.round(F.avg(F.col("is_promoted").cast("double")), 4).alias("promoted_gig_share"),
    )
    .orderBy(F.desc("gig_count")))


def to_text(array_col):
    return F.concat_ws(", ", array_col)


final_df = final_build.select(
    # ---- identity and classification
    "gig_id", "gig_url", "title", "category", "sub_category", "nested_sub_category",
    "service_family",
    to_text("service_labels").alias("service_labels"),
    "classification_confidence", "needs_manual_review",
    # ---- listing and content
    "listing_position", "listing_page", "is_promoted", "is_pro", "is_handpicked",
    "description_word_count", "title_word_count", "tag_count", "technology_count",
    "image_count", "video_count", "portfolio_project_count",
    # ---- demand
    "gig_rating", "gig_review_count", "log_review_count", "orders_in_queue", "saved_by_count",
    "demand_percentile", "gig_review_band",
    # ---- pricing and packages
    "gig_min_price", "gig_max_price", "gig_price_spread",
    "basic_price", "standard_price", "premium_price", "median_package_price",
    "basic_delivery_hours", "standard_delivery_hours", "premium_delivery_hours",
    "basic_revisions", "standard_revisions", "premium_revisions",
    "custom_extra_count", "consultation_price", "package_price_ratio",
    "starting_price",
    # ---- review-sample buyer intelligence
    "sampled_review_count", "avg_review_rating", "median_review_rating", "positive_review_share",
    "business_buyer_share", "repeat_buyer_share", "median_order_duration_days",
    "review_order_price_midpoint_median", "buyer_price_200_plus_share",
    "top_buyer_country", "top_buyer_industry", "latest_sampled_review_date",
    "sampled_review_recency_days", "review_coverage_ratio",
    # ---- competition
    "category_gig_count", "category_seller_count", "category_review_median",
    "category_review_p75", "category_review_p90", "category_price_median",
    "category_promoted_share", "category_top10_review_share",
    "gig_review_percentile", "gig_price_vs_category_median",
    "service_family_gig_count", "service_family_seller_count",
    "service_family_review_median", "service_family_price_median", "service_family_promoted_share",
    # ---- technology and skills
    to_text("technology_list_arr").alias("technology_list"),
    to_text("programming_language_list_arr").alias("programming_language_list"),
    to_text("analysis_type_list_arr").alias("analysis_type_list"),
    to_text("expertise_list_arr").alias("expertise_list"),
    to_text("tools_list_arr").alias("tools_list"),
    to_text("tag_list_arr").alias("tag_list"),
    "python_flag", "sql_flag", "excel_flag", "powerbi_flag", "tableau_flag", "spark_flag",
    "kafka_flag", "dbt_flag", "snowflake_flag", "machine_learning_flag", "automation_flag",
    "dashboard_flag", "data_pipeline_flag", "financial_analytics_flag",
    "marketing_analytics_flag", "product_analytics_flag",
    # ---- seller
    "seller_id", "seller_username", "seller_country", "seller_level", "seller_member_since",
    "seller_tenure_months", "seller_response_time_hours", "seller_hourly_rate",
    "seller_rating", "seller_ratings_count", "seller_completed_orders", "seller_gig_count",
    # ---- snapshot and opportunity scores
    "snapshot_date",
    "demand_score", "competition_score", "pricing_power_score", "business_buyer_score",
    "market_opportunity_score",
)

print(f"final_df: {final_df.count()} rows, {len(final_df.columns)} columns")

# Sample of the transformed feature table
display(final_df.select(
    "gig_id", "title", "service_family", "gig_review_count", "gig_review_band",
    "basic_price", "premium_price", "package_price_ratio",
    "business_buyer_share", "top_buyer_country", "top_buyer_industry",
    "demand_score", "market_opportunity_score",
).limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


final_build: 294 rows, 124 columns
final_df: 294 rows, 115 columns


gig_id,title,service_family,gig_review_count,gig_review_band,basic_price,premium_price,package_price_ratio,business_buyer_share,top_buyer_country,top_buyer_industry,demand_score,market_opportunity_score
11472347,"create custom excel dashboards, trackers, and reports",BI & Dashboarding,575,500+,25.0,200.0,8.0,0.4,United States,financial_services_or_banking,0.993,0.735
110678455,do custom training qlik,Data Consultation & Tutoring,4,1-10,200.0,600.0,3.0,0.0,United States,technology,0.215,0.45
116697469,"do powerful tableau, powerbi, hex, and omni analytics with sql data modeling",BI & Dashboarding,275,101-500,25.0,500.0,20.0,0.0,United States,technology,0.983,0.736
122062614,"do data processing automation, dashboards and system integrations",BI & Dashboarding,6,1-10,250.0,1000.0,4.0,0.2,Brazil,financial_services_or_banking,0.311,0.642
125318834,build custom mixpanel dashboards for product and growth teams,Product & Customer Analytics,26,11-50,380.0,1400.0,3.684,0.0,United States,education,0.761,0.743
132792716,deliver expert data analysis and professional reporting,Statistical & Research Analysis,504,500+,100.0,250.0,2.5,0.0,United States,technology,0.99,0.688
139181195,"do your data analysis in r, spss or python with a clear report",Statistical & Research Analysis,275,101-500,45.0,350.0,7.778,0.0,United States,education,0.983,0.636
140129240,help with process improvement project,Statistical & Research Analysis,19,11-50,50.0,130.0,2.6,0.2,United States,technology,0.648,0.672
141425743,"help spss r analyses, correlation, linear and logistic regression, t test, anova",Statistical & Research Analysis,8,1-10,60.0,null,null,0.0,United States,education,0.392,0.421
143956683,"do econometrics and statical data analysis, report by spss,r ,excel ,python",Statistical & Research Analysis,54,51-100,15.0,200.0,13.333,0.8,United States,construction,0.877,0.644


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Most important derived columns

| Column | Definition |
|---|---|
| `service_family` / `service_labels` | primary / all matched service families (keyword rules over title, tags, description, metadata, nested category) |
| `classification_confidence` | share of the 5 evidence sources that matched the primary family |
| `gig_review_count` | total ratings count — **historical demand proxy**, not orders sold |
| `demand_percentile`, `gig_review_band` | population percentile rank / fixed review bands (0, 1–10, 11–50, 51–100, 101–500, 500+) |
| `gig_min_price`, `gig_max_price`, `package_price_ratio` | advertised price range and premium ÷ basic listing price |
| `sampled_review_count` and all `*_share` review columns | **SAMPLED review metrics only** — not all transactions |
| `review_order_price_midpoint_median` | median estimated order-price midpoint parsed from sampled review ranges (unknowns stay null) |
| `review_coverage_ratio` | sampled (loaded) reviews ÷ total review count per gig |
| `category_*` | nested-sub-category-scoped aggregates (gig count, distinct sellers, review quantiles, promoted share, top-10 concentration) |
| `*_flag` | keyword-based technology / service indicators |
| `seller_gig_count` | gigs listed by the same seller (distinct username) |
| `*_score`, `market_opportunity_score` | transparent percentile components and the documented weighted heuristic — **not** a probability of success |

## 11. Validation

Before writing: unique gig ids equal output rows; no duplicates; row count reconciles with the raw sources; null checks on key columns; critical columns present with correct types.

In [0]:
n_raw = raw_csv.count()
n_final = final_df.count()
n_distinct = final_df.select("gig_id").distinct().count()
n_dupes = final_df.groupBy("gig_id").count().where(F.col("count") > 1).count()

null_counts = {c: final_df.where(F.col(c).isNull()).count()
               for c in ["gig_id", "title", "service_family", "gig_review_count"]}

CRITICAL_COLUMNS = [
    "gig_id", "service_family", "service_labels", "gig_review_count", "gig_review_band",
    "demand_percentile", "basic_price", "premium_price", "package_price_ratio",
    "sampled_review_count", "business_buyer_share", "top_buyer_country",
    "category_gig_count", "gig_price_vs_category_median", "python_flag",
    "seller_gig_count", "market_opportunity_score", "technology_list",
]
missing_cols = [c for c in CRITICAL_COLUMNS if c not in final_df.columns]

EXPECTED_TYPES = {
    "gig_id": ("bigint", "long"),
    "gig_review_count": ("bigint", "long"),
    "basic_price": ("double",),
    "service_family": ("string",),
    "needs_manual_review": ("boolean",),
    "latest_sampled_review_date": ("date",),
}
actual_types = dict(final_df.dtypes)
type_errors = [f"{c}={actual_types.get(c)}" for c, expected in EXPECTED_TYPES.items()
               if actual_types.get(c) not in expected]

validation_rows = [
    ("unique gig ids equal output rows", str(n_distinct), str(n_final),
     "PASS" if n_distinct == n_final else "FAIL"),
    ("no duplicate gig ids", "0", str(n_dupes), "PASS" if n_dupes == 0 else "FAIL"),
    ("output rows match raw CSV record count", str(n_raw), str(n_final),
     "PASS" if n_final == n_raw else "FAIL"),
    ("no nulls in key columns (gig_id/title/service_family/gig_review_count)",
     "0", str(sum(null_counts.values())),
     "PASS" if sum(null_counts.values()) == 0 else "REVIEW"),
    ("all critical columns present", "0 missing",
     f"{len(missing_cols)} missing: {missing_cols}" if missing_cols else "0 missing",
     "PASS" if not missing_cols else "FAIL"),
    ("critical column data types valid", "none", str(type_errors or "none"),
     "PASS" if not type_errors else "FAIL"),
]
validation_report = spark.createDataFrame(validation_rows,
    "check STRING, expected STRING, actual STRING, status STRING")
display(validation_report)

assert n_dupes == 0, "duplicate gig_ids found"
assert n_distinct == n_final, "gig_id uniqueness violated"
assert n_final == n_raw, "row count does not reconcile with raw"
assert not missing_cols, f"missing columns: {missing_cols}"
assert not type_errors, f"type errors: {type_errors}"
print("ALL CRITICAL VALIDATION CHECKS PASSED")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data 

check,expected,actual,status
unique gig ids equal output rows,294,294,PASS
no duplicate gig ids,0,0,PASS
output rows match raw CSV record count,294,294,PASS
no nulls in key columns (gig_id/title/service_family/gig_review_count),0,0,PASS
all critical columns present,0 missing,0 missing,PASS
critical column data types valid,none,none,PASS


ALL CRITICAL VALIDATION CHECKS PASSED


## 12. Delta table creation

Writes the main feature table and the auxiliary technology metrics table as Delta, adds catalog comments to important derived columns, and (optionally) exports a CSV copy to a **new** path — the raw inputs are never modified.

In [0]:
# --- Main feature table (one row per gig) ----------------------------------
(final_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUTPUT_TABLE))

# --- Technology-level aggregate table --------------------------------------
(tech_metrics.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TECH_METRICS_TABLE))

# --- Optional CSV export (new path - raw inputs never touched) --------------
if EXPORT_CSV:
    (final_df.coalesce(1).write
        .mode("overwrite")
        .option("header", True)
        .option("quoteAll", True)
        .csv(CSV_EXPORT_PATH))
    print(f"CSV export written to: {CSV_EXPORT_PATH}")

# --- Post-write sanity check ------------------------------------------------
display(spark.sql(f"""
    SELECT COUNT(*) AS rows_written,
           COUNT(DISTINCT gig_id) AS distinct_gig_ids,
           COUNT(DISTINCT seller_username) AS distinct_sellers
    FROM {OUTPUT_TABLE}
"""))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


CSV export written to: /Volumes/demo/bronze/fiverr/exports/fiverr_gig_market_features_csv


rows_written,distinct_gig_ids,distinct_sellers
294,294,250


In [0]:
%sql
-- Catalog documentation for important derived columns
COMMENT ON TABLE demo.market_intelligence.fiverr_gig_market_features IS 'Fiverr Data Analytics gig snapshot (scraped 2026-11-15). One row per gig. Review-derived buyer/price metrics are SAMPLE-BASED; review counts are a demand proxy. Single snapshot - no trend claims.';

ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN service_family COMMENT 'Primary service family from multi-label keyword rules (title, tags, description, metadata, nested category; priority-ordered).';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN service_labels COMMENT 'All matching service families, comma-separated (multi-label).';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN classification_confidence COMMENT 'Share of 5 evidence sources matching the primary family keywords.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN needs_manual_review COMMENT 'True when no family matched or classification_confidence < 0.4.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN gig_review_count COMMENT 'Total gig ratings count - HISTORICAL DEMAND PROXY, not exact orders sold.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN demand_percentile COMMENT 'Percentile rank of gig review count across the whole snapshot.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN gig_review_band COMMENT 'Review bands: 0 reviews, 1-10, 11-50, 51-100, 101-500, 500+.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN package_price_ratio COMMENT 'Premium package listing price / basic package listing price (not transaction prices).';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN sampled_review_count COMMENT 'Number of sampled review records available for this gig (sample only).';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN business_buyer_share COMMENT 'Share of SAMPLED reviews flagged is_business - sample only, not all transactions.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN repeat_buyer_share COMMENT 'Share of SAMPLED reviews from repeat buyers - sample only.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN review_order_price_midpoint_median COMMENT 'Median estimated order price midpoint parsed from sampled review price ranges; unknowns stay null.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN review_coverage_ratio COMMENT 'Sampled (loaded) reviews divided by total review count for the gig.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN category_top10_review_share COMMENT 'Share of nested-sub-category review demand held by the top 10 gigs by review count.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN gig_price_vs_category_median COMMENT 'Starting price divided by nested-sub-category median starting price.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN demand_score COMMENT 'Percentile rank of log review count (demand proxy component).';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN competition_score COMMENT 'Percentile rank of nested-sub-category gig count (higher = more crowded).';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN pricing_power_score COMMENT 'Percentile rank of gig price vs category median.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN business_buyer_score COMMENT 'Percentile rank of sampled business-buyer share; null when no sampled reviews.';
ALTER TABLE demo.market_intelligence.fiverr_gig_market_features ALTER COLUMN market_opportunity_score COMMENT 'Heuristic: 0.40*demand + 0.25*(1-competition) + 0.20*pricing_power + 0.15*business_buyer. NOT an objective probability of success.';

## 13. Dashboard SQL queries

Standalone queries for a Databricks SQL Dashboard, one block per dashboard section:

* **A** Market Overview · **B** Demand Analysis · **C** Competition Analysis · **D** Pricing Analysis · **E** Buyer Intelligence · **F** Technology Demand · **G** Gig Explorer

Labels containing “sampled” reflect the review sample only. This is a single snapshot — no growth/decline claims.

In [0]:
%sql
-- A. Market Overview
SELECT
  COUNT(*) AS gigs,
  COUNT(DISTINCT seller_username) AS unique_sellers,
  ROUND(AVG(gig_rating), 2) AS avg_gig_rating,
  PERCENTILE_APPROX(gig_review_count, 0.5) AS median_gig_review_count,
  PERCENTILE_APPROX(starting_price, 0.5) AS median_starting_price_usd,
  ROUND(AVG(business_buyer_share) * 100, 1) AS business_buyer_share_pct_sampled,
  ROUND(AVG(repeat_buyer_share) * 100, 1) AS repeat_buyer_share_pct_sampled
FROM demo.market_intelligence.fiverr_gig_market_features

gigs,unique_sellers,avg_gig_rating,median_gig_review_count,median_starting_price_usd,business_buyer_share_pct_sampled,repeat_buyer_share_pct_sampled
294,250,4.72,11,50.0,14.1,39.1


In [0]:
%sql
-- B1. Demand Analysis - review demand by service family
SELECT
  service_family,
  COUNT(*) AS gigs,
  SUM(gig_review_count) AS total_reviews,
  PERCENTILE_APPROX(gig_review_count, 0.5) AS median_reviews,
  PERCENTILE_APPROX(gig_review_count, 0.9) AS p90_reviews,
  MAX(latest_sampled_review_date) AS most_recent_sampled_review
FROM demo.market_intelligence.fiverr_gig_market_features
GROUP BY service_family
ORDER BY total_reviews DESC

service_family,gigs,total_reviews,median_reviews,p90_reviews,most_recent_sampled_review
Statistical & Research Analysis,171,8234,15,88,2026-11-14
BI & Dashboarding,69,1886,9,55,2026-11-14
Product & Customer Analytics,8,173,9,81,2026-11-07
Financial & Business Analytics,11,145,6,41,2026-11-02
Marketing & Web Analytics,10,105,3,24,2026-09-04
Data Science & Machine Learning,12,92,4,12,2026-11-04
Supply Chain & Operations Analytics,4,75,12,39,2026-06-11
Data Consultation & Tutoring,6,63,4,27,2026-11-14
Data Engineering & Data Pipelines,2,25,7,18,2026-05-07
Data Cleaning & Processing,1,5,5,5,2024-12-04


In [0]:
%sql
-- B2. Top 20 gigs by review count (demand proxy)
SELECT title, gig_url, service_family, seller_level,
       gig_review_count, gig_rating, starting_price
FROM demo.market_intelligence.fiverr_gig_market_features
ORDER BY gig_review_count DESC
LIMIT 20

title,gig_url,service_family,seller_level,gig_review_count,gig_rating,starting_price
provide real estate property title search and surplus funds all over the USA,https://www.fiverr.com/abrhamjoy45/provide-real-estate-property-title-search-and-surplus-funds-all-over-the-usa,Statistical & Research Analysis,LEVEL_TWO,1690,5.0,5.0
"do econometrics and statistical data analysis with spss, stata, r, python, excel",https://www.fiverr.com/xarimstudio/do-amazing-eye-catching-logo-animation-intros-in-4k,Statistical & Research Analysis,LEVEL_TWO,1165,4.7,20.0
"create custom excel dashboards, trackers, and reports",https://www.fiverr.com/indigocraft/help-you-with-excel-spreadsheets,BI & Dashboarding,LEVEL_TWO,575,5.0,25.0
deliver expert data analysis and professional reporting,https://www.fiverr.com/naimul_sust/do-data-cleaning-visualization-modeling-and-report-making,Statistical & Research Analysis,Top Rated Seller,504,4.9,100.0
"do your data analysis in r, spss or python with a clear report",https://www.fiverr.com/rafreis/do-statistical-data-analysis-using-spss-amos-or-excel,Statistical & Research Analysis,Top Rated Seller,275,4.9,45.0
"do powerful tableau, powerbi, hex, and omni analytics with sql data modeling",https://www.fiverr.com/aliraza95/do-stunning-tableau-dashboards-maps-etc,BI & Dashboarding,LEVEL_ONE,275,5.0,125.0
help in machine learning and statistical data analysis in rstudio r,https://www.fiverr.com/m_faizan_ak/help-in-machine-learning-and-data-analysis-in-rstudio-r,Statistical & Research Analysis,No level,257,4.9,40.0
do medical research data analysis and statistical data analysis using spss,https://www.fiverr.com/demigod299/do-expert-data-analysis-and-reporting-spss-stata-minitab,Statistical & Research Analysis,Top Rated Seller,198,5.0,25.0
"do anything related to python, pandas, numpy, matplotlib, jupyter",https://www.fiverr.com/m_harriisss/analyze-your-data-and-create-power-bi-reports-and-dashboards,Statistical & Research Analysis,LEVEL_TWO,154,5.0,65.0
"teach sql, excel, google sheets and python",https://www.fiverr.com/faiquaimran/teach-sql-excel-and-google-sheets,BI & Dashboarding,LEVEL_TWO,148,5.0,10.0


In [0]:
%sql
-- B3. Demand distribution - gigs by review band
SELECT gig_review_band,
       COUNT(*) AS gigs,
       SUM(gig_review_count) AS total_reviews_in_band
FROM demo.market_intelligence.fiverr_gig_market_features
GROUP BY gig_review_band
ORDER BY MIN(gig_review_count)

gig_review_band,gigs,total_reviews_in_band
0 reviews,9,0
1-10,137,652
11-50,110,2547
51-100,21,1470
101-500,13,2200
500+,4,3934


In [0]:
%sql
-- C1. Competition - category overview (nested sub-category)
SELECT
  nested_sub_category AS category,
  MAX(category_gig_count) AS gig_count,
  MAX(category_seller_count) AS unique_sellers,
  SUM(gig_review_count) AS total_reviews,
  MAX(category_review_median) AS median_reviews_per_gig,
  MAX(category_review_p90) AS p90_reviews_per_gig,
  ROUND(MAX(category_review_median) / MAX(category_gig_count), 1) AS demand_per_gig_ratio,
  ROUND(MAX(category_promoted_share) * 100, 1) AS promoted_share_pct,
  ROUND(MAX(category_top10_review_share) * 100, 1) AS top10_review_concentration_pct
FROM demo.market_intelligence.fiverr_gig_market_features
GROUP BY nested_sub_category
ORDER BY total_reviews DESC

category,gig_count,unique_sellers,total_reviews,median_reviews_per_gig,p90_reviews_per_gig,demand_per_gig_ratio,promoted_share_pct,top10_review_concentration_pct
Statistical Modeling & Analytics,140,121,7247,15.0,83.0,0.1,30.7,61.3
Data Analytics Consultation,65,61,2243,9.0,70.0,0.1,13.9,73.9
BI Analytics,33,33,471,7.0,23.0,0.2,3.0,75.0
Financial Analytics,28,27,410,11.0,41.0,0.4,3.6,71.2
Marketing Analytics,9,9,203,6.0,143.0,0.7,0.0,100.0
Product Analytics,9,8,142,9.0,81.0,1.0,33.3,100.0
Supply Chain Analytics,10,9,87,3.0,15.0,0.3,0.0,100.0


In [0]:
%sql
-- C2. Promoted vs non-promoted gigs
SELECT is_promoted,
       COUNT(*) AS gigs,
       COUNT(DISTINCT seller_username) AS unique_sellers,
       PERCENTILE_APPROX(gig_review_count, 0.5) AS median_reviews,
       PERCENTILE_APPROX(starting_price, 0.5) AS median_starting_price_usd
FROM demo.market_intelligence.fiverr_gig_market_features
GROUP BY is_promoted

is_promoted,gigs,unique_sellers,median_reviews,median_starting_price_usd
false,237,209,9,50.0
true,57,50,23,40.0


In [0]:
%sql
-- D1. Pricing - package prices and starting-price distribution by service family
SELECT
  service_family,
  PERCENTILE_APPROX(basic_price, 0.5) AS median_basic_price,
  PERCENTILE_APPROX(standard_price, 0.5) AS median_standard_price,
  PERCENTILE_APPROX(premium_price, 0.5) AS median_premium_price,
  ROUND(AVG(package_price_ratio), 2) AS avg_premium_to_basic_ratio,
  PERCENTILE_APPROX(starting_price, 0.25) AS starting_price_p25,
  PERCENTILE_APPROX(starting_price, 0.5) AS starting_price_median,
  PERCENTILE_APPROX(starting_price, 0.75) AS starting_price_p75
FROM demo.market_intelligence.fiverr_gig_market_features
GROUP BY service_family
ORDER BY starting_price_median DESC

service_family,median_basic_price,median_standard_price,median_premium_price,avg_premium_to_basic_ratio,starting_price_p25,starting_price_median,starting_price_p75
Data Cleaning & Processing,250.0,750.0,1750.0,7.0,250.0,250.0,250.0
Data Consultation & Tutoring,50.0,100.0,150.0,8.59,100.0,100.0,125.0
BI & Dashboarding,50.0,100.0,250.0,7.77,30.0,65.0,150.0
Financial & Business Analytics,55.0,170.0,350.0,4.5,40.0,55.0,100.0
Marketing & Web Analytics,50.0,100.0,220.0,8.39,25.0,50.0,70.0
Product & Customer Analytics,50.0,100.0,200.0,4.9,40.0,50.0,150.0
Data Science & Machine Learning,40.0,90.0,205.0,6.74,30.0,40.0,50.0
Statistical & Research Analysis,30.0,80.0,150.0,6.22,20.0,40.0,65.0
Supply Chain & Operations Analytics,20.0,40.0,60.0,5.14,20.0,30.0,35.0
Data Engineering & Data Pipelines,30.0,80.0,120.0,4.53,30.0,30.0,50.0


In [0]:
%sql
-- D2. Reported order price-range distribution (from sampled reviews only)
WITH priced AS (
  SELECT review_order_price_midpoint_median AS mid
  FROM demo.market_intelligence.fiverr_gig_market_features
  WHERE review_order_price_midpoint_median IS NOT NULL
)
SELECT
  CASE WHEN mid < 50  THEN 'a. < $50'
       WHEN mid < 100 THEN 'b. $50 - $100'
       WHEN mid < 200 THEN 'c. $100 - $200'
       WHEN mid < 400 THEN 'd. $200 - $400'
       ELSE 'e. $400+' END AS sampled_order_price_band,
  COUNT(*) AS gigs,
  ROUND(AVG(mid), 0) AS avg_reported_midpoint_usd
FROM priced
GROUP BY 1
ORDER BY 1

sampled_order_price_band,gigs,avg_reported_midpoint_usd
a. <,97,25.0
b. -,84,75.0
c. -,63,150.0
d. -,31,300.0
e. +,7,779.0


In [0]:
%sql
-- E1. Buyer intelligence - sampled buyer metrics by service family
SELECT
  service_family,
  COUNT(*) AS gigs_with_sampled_reviews,
  ROUND(AVG(business_buyer_share) * 100, 1) AS business_buyer_share_pct,
  ROUND(AVG(repeat_buyer_share) * 100, 1) AS repeat_buyer_share_pct,
  PERCENTILE_APPROX(median_order_duration_days, 0.5) AS median_order_duration_days
FROM demo.market_intelligence.fiverr_gig_market_features
WHERE sampled_review_count > 0
GROUP BY service_family
ORDER BY business_buyer_share_pct DESC

service_family,gigs_with_sampled_reviews,business_buyer_share_pct,repeat_buyer_share_pct,median_order_duration_days
Data Cleaning & Processing,1,40.0,60.0,14.0
Product & Customer Analytics,8,29.2,17.5,3.0
Data Science & Machine Learning,11,20.0,25.9,4.0
Marketing & Web Analytics,10,18.0,32.3,2.0
Financial & Business Analytics,10,18.0,25.0,3.0
Data Consultation & Tutoring,6,16.7,35.0,1.0
BI & Dashboarding,64,15.4,50.1,3.0
Statistical & Research Analysis,169,12.4,38.1,4.0
Supply Chain & Operations Analytics,4,0.0,35.0,4.0
Data Engineering & Data Pipelines,2,0.0,50.0,6.0


In [0]:
%sql
-- E2. Top buyer countries (most frequent country of each gig's sampled reviews)
SELECT top_buyer_country,
       COUNT(*) AS gigs,
       SUM(gig_review_count) AS total_reviews
FROM demo.market_intelligence.fiverr_gig_market_features
WHERE top_buyer_country IS NOT NULL
GROUP BY top_buyer_country
ORDER BY gigs DESC

top_buyer_country,gigs,total_reviews
United States,161,8035
United Kingdom,35,636
United Arab Emirates,11,308
Germany,10,158
Saudi Arabia,10,446
Italy,5,122
Malaysia,3,35
France,3,17
Israel,3,14
Canada,3,24


In [0]:
%sql
-- E3. Top buyer industries (from gig-level sampled industry counts)
SELECT top_buyer_industry,
       COUNT(*) AS gigs,
       SUM(sampled_review_count) AS sampled_reviews
FROM demo.market_intelligence.fiverr_gig_market_features
WHERE top_buyer_industry IS NOT NULL
GROUP BY top_buyer_industry
ORDER BY gigs DESC

top_buyer_industry,gigs,sampled_reviews
technology,42,209
education,32,158
medical_or_pharmaceutical,14,67
financial_services_or_banking,10,50
marketing_or_advertising,8,35
retail_and_wholesale,8,40
real_estate,6,29
science,5,21
sports_and_fitness,4,20
construction,4,17


In [0]:
%sql
-- F1. Technology demand - frequency, review demand, price, promotion
SELECT technology,
       gig_count,
       median_gig_review_count,
       median_starting_price,
       ROUND(promoted_gig_share * 100, 1) AS promoted_share_pct
FROM demo.market_intelligence.fiverr_technology_metrics
ORDER BY gig_count DESC
LIMIT 25

technology,gig_count,median_gig_review_count,median_starting_price,promoted_share_pct
excel,253,11.0,40.0,20.5
google sheets,182,10.0,50.0,21.4
power bi,164,9.0,50.0,18.3
google analytics,136,10.0,40.0,22.8
stata,131,13.0,40.0,29.8
jupyter notebook,127,10.0,45.0,20.5
tableau,125,11.0,40.0,24.8
spss,113,15.0,35.0,33.6
looker studio,95,10.0,50.0,19.0
sas,82,10.0,40.0,30.5


In [0]:
%sql
-- F2. Popular technology combinations
SELECT 'Python + SQL' AS technology_combination, COUNT(*) AS gigs,
       PERCENTILE_APPROX(gig_review_count, 0.5) AS median_reviews,
       PERCENTILE_APPROX(starting_price, 0.5) AS median_price_usd
FROM demo.market_intelligence.fiverr_gig_market_features WHERE python_flag AND sql_flag
UNION ALL
SELECT 'Power BI + SQL', COUNT(*), PERCENTILE_APPROX(gig_review_count, 0.5), PERCENTILE_APPROX(starting_price, 0.5)
FROM demo.market_intelligence.fiverr_gig_market_features WHERE powerbi_flag AND sql_flag
UNION ALL
SELECT 'Python + Machine Learning', COUNT(*), PERCENTILE_APPROX(gig_review_count, 0.5), PERCENTILE_APPROX(starting_price, 0.5)
FROM demo.market_intelligence.fiverr_gig_market_features WHERE python_flag AND machine_learning_flag
UNION ALL
SELECT 'Excel + Power BI', COUNT(*), PERCENTILE_APPROX(gig_review_count, 0.5), PERCENTILE_APPROX(starting_price, 0.5)
FROM demo.market_intelligence.fiverr_gig_market_features WHERE excel_flag AND powerbi_flag
UNION ALL
SELECT 'Python + Tableau', COUNT(*), PERCENTILE_APPROX(gig_review_count, 0.5), PERCENTILE_APPROX(starting_price, 0.5)
FROM demo.market_intelligence.fiverr_gig_market_features WHERE python_flag AND tableau_flag
ORDER BY median_reviews DESC

technology_combination,gigs,median_reviews,median_price_usd
Python + Tableau,119,11,40.0
Python + SQL,157,10,45.0
Excel + Power BI,151,10,50.0
Power BI + SQL,129,8,50.0
Python + Machine Learning,21,6,40.0


In [0]:
%sql
-- G. Gig Explorer
SELECT
  title,
  gig_url,
  service_family,
  seller_level,
  gig_review_count,
  gig_rating,
  starting_price,
  basic_price,
  standard_price,
  premium_price,
  technology_list,
  is_promoted
FROM demo.market_intelligence.fiverr_gig_market_features
ORDER BY gig_review_count DESC
LIMIT 100

title,gig_url,service_family,seller_level,gig_review_count,gig_rating,starting_price,basic_price,standard_price,premium_price,technology_list,is_promoted
provide real estate property title search and surplus funds all over the USA,https://www.fiverr.com/abrhamjoy45/provide-real-estate-property-title-search-and-surplus-funds-all-over-the-usa,Statistical & Research Analysis,LEVEL_TWO,1690,5.0,5.0,5.0,10.0,15.0,"excel, google analytics, google sheets, other",false
"do econometrics and statistical data analysis with spss, stata, r, python, excel",https://www.fiverr.com/xarimstudio/do-amazing-eye-catching-logo-animation-intros-in-4k,Statistical & Research Analysis,LEVEL_TWO,1165,4.7,20.0,20.0,250.0,550.0,"excel, google sheets, matlab, maxqda, minitab, nvivo, spss, stata",true
"create custom excel dashboards, trackers, and reports",https://www.fiverr.com/indigocraft/help-you-with-excel-spreadsheets,BI & Dashboarding,LEVEL_TWO,575,5.0,25.0,25.0,75.0,200.0,"excel, google sheets, vba",false
deliver expert data analysis and professional reporting,https://www.fiverr.com/naimul_sust/do-data-cleaning-visualization-modeling-and-report-making,Statistical & Research Analysis,Top Rated Seller,504,4.9,100.0,100.0,150.0,250.0,"excel, google sheets, power bi, sas, spss, stata, tableau",true
"do your data analysis in r, spss or python with a clear report",https://www.fiverr.com/rafreis/do-statistical-data-analysis-using-spss-amos-or-excel,Statistical & Research Analysis,Top Rated Seller,275,4.9,45.0,45.0,225.0,350.0,"google analytics, jupyter notebook, other, spss, stata",true
"do powerful tableau, powerbi, hex, and omni analytics with sql data modeling",https://www.fiverr.com/aliraza95/do-stunning-tableau-dashboards-maps-etc,BI & Dashboarding,LEVEL_ONE,275,5.0,125.0,25.0,125.0,500.0,"bigquery, excel, google analytics, google sheets, looker studio, mysql, pandas, power bi, spark, tableau",false
help in machine learning and statistical data analysis in rstudio r,https://www.fiverr.com/m_faizan_ak/help-in-machine-learning-and-data-analysis-in-rstudio-r,Statistical & Research Analysis,No level,257,4.9,40.0,40.0,75.0,140.0,"jupyter notebook, other",false
do medical research data analysis and statistical data analysis using spss,https://www.fiverr.com/demigod299/do-expert-data-analysis-and-reporting-spss-stata-minitab,Statistical & Research Analysis,Top Rated Seller,198,5.0,25.0,25.0,200.0,300.0,"excel, google analytics, google sheets, other, stata",false
"do anything related to python, pandas, numpy, matplotlib, jupyter",https://www.fiverr.com/m_harriisss/analyze-your-data-and-create-power-bi-reports-and-dashboards,Statistical & Research Analysis,LEVEL_TWO,154,5.0,65.0,20.0,65.0,125.0,"google sheets, jupyter notebook",false
"teach sql, excel, google sheets and python",https://www.fiverr.com/faiquaimran/teach-sql-excel-and-google-sheets,BI & Dashboarding,LEVEL_TWO,148,5.0,10.0,10.0,20.0,30.0,"excel, google analytics, google sheets, other, power bi, tableau",false
